# DATA 620
## Document Classification
### Kevin Havis & Aaliyah John-Harry

## Introduction

### Plan:
1. Dataset used
2. Preprocessing to be done
3. Models to be used

## Data Loading

### Plan:
1. Load dataset
2. Show:
    - Shape (# rows, columns)
    - Class distribution (spam vs ham)
3. Inspect sample emails

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("spam_ham_dataset.csv", index_col=0)

print(df.shape)
print(df["label"].value_counts())
df.head(3)

## Data Preprocessing

### Plan:
1. Lowercasing
2. Removing punctuation
3. Removing stopwords
4. Tokenization...

In [ ]:
import re
import string
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
STOPWORDS = set(stopwords.words("english"))

def preprocess(text):
    text = text.lower()
    text = re.sub(r"subject:\s*", "", text)   # strip email subject prefix
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS and t.isalpha()]
    return " ".join(tokens)

df["clean_text"] = df["text"].apply(preprocess)
df[["text", "clean_text"]].head(3)

## Feature Engineering

### Plan:
1. Convert text to numerical features.
    - CountVectorizer/TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=10_000)
X = vectorizer.fit_transform(df["clean_text"])
y = df["label_num"]

print(f"Feature matrix: {X.shape}")

## Train/Test Split

### Plan:
1. 80/20 dataset split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}")

## Model Building

### Possible models:
1. Naive Bayes
2. Logistic Regression
3. RF

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

for name, model in models.items():
    model.fit(X_train, y_train)

## Model Evaluation

### Plan:
1. Accuracy
2. Precision/Recall/F1-score
3. Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

results = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    results[name] = report
    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=["ham", "spam"]))

# Confusion matrices
fig, axes = plt.subplots(1, len(models), figsize=(14, 4))
for ax, (name, model) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, model.predict(X_test))
    ConfusionMatrixDisplay(cm, display_labels=["ham", "spam"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

## Results & Comparison

### Plan:
1. Compare models
2. Identify the best model

In [ ]:
metrics = ["precision", "recall", "f1-score"]
summary = pd.DataFrame(
    {name: {m: results[name]["weighted avg"][m] for m in metrics} for name in results}
).T

print(summary.to_string())

summary.plot(kind="bar", figsize=(8, 4), ylim=(0.9, 1.0))
plt.title("Model Comparison (weighted avg)")
plt.ylabel("Score")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

best = summary["f1-score"].idxmax()
print(f"\nBest model by F1: {best}")

## Conclusion

### Plan:
1. Summary
2. Future Improvements